# Evaluation engine — compositional image retrieval

This notebook is the **ruler** of the project: every method (Tier-0 baseline, CLAY,
our fusion module) is scored through the functions here, so they all get measured the
same way.

Metric definitions are taken **verbatim** from the assignment spec
(`documents/project_specification.md`, section 3.1.3):

- **Recall@K (hit rate)** — the *primary* metric. `1` if the top-K share at least one
  image with the ground-truth set `G`, else `0`. (This is a hit-rate, **not** textbook
  recall `|hits|/|relevant|`.)
- **Precision@K** — secondary. `|top-K ∩ G| / K`.

Both are computed per source image, then **averaged over all valid sources** of a query.

All image identifiers are **dataset indices** (ints), per `CONTRACT.md` §0. A *ranking*
is a `list[int]` of dataset indices, best match first, with the source image already
removed (`CONTRACT.md` §5).

**How to use this notebook:** Run all cells top-to-bottom. The last section is a
self-test that proves the metrics are correct *without* needing CLIP or the attribute
tensor — so you can trust the ruler before any real method exists.

## 1. Imports & locating the evaluation JSON

In [ ]:
from __future__ import annotations

import json
from pathlib import Path


def find_eval_json(start: Path | None = None) -> Path:
    """Walk upward from the current dir to find Evaluation/celeba_evaluation.json.

    Works whether the notebook is run from src/, the repo root, or Colab (where you
    set EVAL_JSON manually if needed).
    """
    here = (start or Path.cwd()).resolve()
    for folder in [here, *here.parents]:
        candidate = folder / 'Evaluation' / 'celeba_evaluation.json'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find Evaluation/celeba_evaluation.json; set the path manually.')


EVAL_JSON = find_eval_json()
print('Using evaluation JSON:', EVAL_JSON)

## 2. Per-source metrics

The two atomic functions. Everything else just averages these.

In [ ]:
def recall_at_k(ranking: list[int], ground_truth: set[int], k: int) -> float:
    """Hit rate: 1.0 if any ground-truth image is in the top-K, else 0.0."""
    topk = ranking[:k]
    return 1.0 if (set(topk) & ground_truth) else 0.0


def precision_at_k(ranking: list[int], ground_truth: set[int], k: int) -> float:
    """Fraction of the top-K that are ground-truth images (denominator is K)."""
    topk = ranking[:k]
    hits = len(set(topk) & ground_truth)
    return hits / k

## 3. Per-query and full-benchmark scoring

`evaluate_query` averages the metrics over all valid sources of one query.
`evaluate_all` runs every query in the JSON. A retrieval *method* plugs in through the
`get_ranking(source_idx) -> ranking` callback — fake rankings for testing now, real
CLIP-based rankings later.

In [ ]:
def evaluate_query(ground_truth: dict, get_ranking, ks=(1, 5, 10)) -> dict:
    """Score one query, averaging each metric over all of its valid source images."""
    sums = {f'recall@{k}': 0.0 for k in ks}
    sums.update({f'precision@{k}': 0.0 for k in ks})

    sources = list(ground_truth.keys())
    for src in sources:
        src_idx = int(src)                       # keys may be strings -> int
        targets = {int(t) for t in ground_truth[src]}
        ranking = get_ranking(src_idx)
        for k in ks:
            sums[f'recall@{k}'] += recall_at_k(ranking, targets, k)
            sums[f'precision@{k}'] += precision_at_k(ranking, targets, k)

    n = len(sources)
    results = {m: (total / n if n else 0.0) for m, total in sums.items()}
    results['num_sources'] = n
    return results


def evaluate_all(gt_list: list[dict], make_get_ranking, ks=(1, 5, 10)) -> dict:
    """Score every query in the evaluation JSON.

    `make_get_ranking(query_str)` returns a get_ranking function for that query.
    """
    out = {}
    for entry in gt_list:
        query_str = entry['query']
        get_ranking = make_get_ranking(query_str)
        out[query_str] = evaluate_query(entry['ground_truth'], get_ranking, ks)
    return out

## 4. Helpers — load JSON, parse queries, pretty table

In [ ]:
def load_eval_json(path) -> list[dict]:
    """Load the authoritative evaluation JSON (a list of query dicts)."""
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def parse_query(query_str: str) -> tuple[list[str], list[str]]:
    """Turn a query string into (positive_attrs, negative_attrs).

    Handles JSON style ('+Black_Hair, -Wavy_Hair') and spec-table style
    ('+ Black Hair & - Wavy Hair'). Names use underscores, e.g. 'Black_Hair'.
    """
    pos, neg = [], []
    for piece in query_str.replace('&', ',').split(','):
        piece = piece.strip()
        if not piece:
            continue
        sign, rest = piece[0], piece[1:].strip()
        name = rest.replace(' ', '_')
        if sign == '+':
            pos.append(name)
        elif sign == '-':
            neg.append(name)
        else:
            raise ValueError(f'Query piece must start with + or -: {piece!r}')
    return pos, neg


def format_results_table(all_results: dict, ks=(1, 5, 10)) -> str:
    """Render evaluate_all() output as a readable text table."""
    cols = ['query'] + [f'R@{k}' for k in ks] + [f'P@{k}' for k in ks] + ['#src']
    rows = [cols]
    for query, res in all_results.items():
        rows.append(
            [query]
            + [f"{res[f'recall@{k}']:.3f}" for k in ks]
            + [f"{res[f'precision@{k}']:.3f}" for k in ks]
            + [str(res['num_sources'])]
        )
    widths = [max(len(r[i]) for r in rows) for i in range(len(cols))]
    lines = []
    for r, row in enumerate(rows):
        lines.append('  '.join(cell.ljust(widths[i]) for i, cell in enumerate(row)))
        if r == 0:
            lines.append('  '.join('-' * w for w in widths))
    return '\n'.join(lines)

## 5. Self-test — prove the ruler is correct (no CLIP, no attributes)

### 5a. Unit tests on the two core functions

In [ ]:
gt = {2, 4, 6, 8, 10}
assert recall_at_k([1, 2, 3], gt, 1) == 0.0, 'no hit in top-1 -> 0'
assert recall_at_k([2, 1, 3], gt, 1) == 1.0, 'hit at rank 1 -> 1'
assert recall_at_k([1, 3, 2], gt, 5) == 1.0, 'hit within top-5 -> 1'
assert recall_at_k([1, 3, 5], gt, 5) == 0.0, 'no hit in top-5 -> 0'
assert precision_at_k([2, 4, 1, 3, 5], gt, 5) == 2 / 5, '2 of 5 are GT'
assert precision_at_k([2, 4, 6, 8, 10], gt, 5) == 1.0, 'all 5 are GT'
assert precision_at_k([1, 3, 5, 7, 9], gt, 5) == 0.0, 'none are GT'
print('[PASS] unit tests on recall_at_k / precision_at_k')

### 5b. Realistic tests against the actual evaluation JSON

Two fake methods bracket the possible scores:
- **Oracle** returns the ground-truth targets first → Recall must be **1.0** everywhere.
- **Adversary** returns only non-GT indices → Recall and Precision must be **0.0**.

In [ ]:
gt_list = load_eval_json(EVAL_JSON)
print(f'Loaded {len(gt_list)} queries from {EVAL_JSON.name}')


def _non_gt_indices(targets, n=10):
    """Return n indices that are NOT in `targets` (for the adversary test)."""
    bad = {int(t) for t in targets}
    out, i = [], 0
    while len(out) < n:
        if i not in bad:
            out.append(i)
        i += 1
    return out


for entry in gt_list:
    q, g = entry['query'], entry['ground_truth']

    oracle = lambda src, g=g: [int(t) for t in g[str(src)]]
    res = evaluate_query(g, oracle)
    assert res['recall@1'] == 1.0, f'oracle should hit@1 on {q!r}'
    assert res['recall@5'] == 1.0 and res['recall@10'] == 1.0
    assert res['precision@1'] == 1.0, f'oracle top-1 is a GT on {q!r}'

    adversary = lambda src, g=g: _non_gt_indices(g[str(src)])
    res_bad = evaluate_query(g, adversary)
    assert res_bad['recall@10'] == 0.0, f'adversary must miss on {q!r}'
    assert res_bad['precision@10'] == 0.0

print('[PASS] oracle scores 1.0 and adversary scores 0.0 on ALL', len(gt_list), 'queries')

### 5c. Eyeball the oracle table (the upper bound)

Recall is 1.000 everywhere. Note **P@10 dips below 1.0** for rare/composed queries
(e.g. `+Mustache`) — because those sources have fewer than 10 valid targets, so even a
perfect method can't fill 10 slots with hits. This is a property of Precision@K, not a
bug, and it's why Recall@K is the primary metric.

In [ ]:
def make_oracle_for(query_str):
    g = next(e['ground_truth'] for e in gt_list if e['query'] == query_str)
    return lambda src, g=g: [int(t) for t in g[str(src)]]


all_res = evaluate_all(gt_list, make_oracle_for)
print(format_results_table(all_res))

### 5d. `parse_query` sanity

In [ ]:
assert parse_query('+Smiling') == (['Smiling'], [])
assert parse_query('+Black_Hair, -Wavy_Hair') == (['Black_Hair'], ['Wavy_Hair'])
assert parse_query('+ Eyeglasses & - Smiling') == (['Eyeglasses'], ['Smiling'])
print('[PASS] parse_query handles JSON style and spec-table style')
print('\nALL TESTS PASSED. The ruler is trustworthy.')